In [1]:
import stanza
nlp = stanza.Pipeline(lang="sa",processors = "tokenize,pos")
import re

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-28 00:25:12 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-28 00:25:12 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-06-28 00:25:13 INFO: Loading these models for language: sa (Sanskrit):
| Processor | Package        |
------------------------------
| tokenize  | vedic          |
| pos       | vedic_nocharlm |

2026-06-28 00:25:13 INFO: Using device: cpu
2026-06-28 00:25:13 INFO: Loading: tokenize
2026-06-28 00:25:15 INFO: Loading: pos
2026-06-28 00:25:16 INFO: Done loading processors!


In [1]:
import re
import unicodedata
import stanza

# 1. Download the specific classical treebank model package
stanza.download("sa", package="itb")

# 2. Configure each neural layer manually to use the ITB engine
nlp = stanza.Pipeline(
    lang="sa",
    processors={
        "tokenize": "itb",
        "pos": "itb",
        "lemma": "itb",
        "depparse": "itb",
    },
)


def process_pure_stanza(text: str):
    if not text:
        return []

    # Step A: Clean up hidden characters and normalize Unicode to NFC format
    cleaned = unicodedata.normalize("NFC", str(text))
    cleaned = re.sub(r"[\u200B-\u200D]", "", cleaned)

    # Step B: CRITICAL FOR STANZA ACCURACY
    # Pad dandas with spaces so the neural tokenizer doesn't merge characters
    cleaned = re.sub(r"([।॥])", r" \1 ", cleaned)

    # Collapse any resulting duplicate spaces
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Step C: Pipe straight into Stanza
    doc = nlp(cleaned)

    results = []
    for sentence in doc.sentences:
        for word in sentence.words:
            results.append(
                {"token": word.text, "lemma": word.lemma, "pos": word.upos}
            )
    return results


# =========================================================
# Executing and Verification
# =========================================================
sample_text = "शशकः वनम् अगच्छत् ।"
parsed_tokens = process_pure_stanza(sample_text)

for item in parsed_tokens:
    print(
        f"Token: {item['token']:<12} | Lemma (Root): {item['lemma']:<12} | POS: {item['pos']}"
    )

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-28 01:43:09 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-06-28 01:43:09 WARNING: Can not find package: itb.
2026-06-28 01:43:09 INFO: Downloading these customized packages for language: sa (Sanskrit)...
| Processor | Package |
-----------------------

2026-06-28 01:43:09 INFO: Finished downloading models and saved to /root/.cache/stanza/1.13.0/resources
2026-06-28 01:43:09 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-28 01:43:09 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-06-28 01:43:09 WARNING: Can not find t

Token: शशकः         | Lemma (Root): शशकः         | POS: ADV
Token: वनम्         | Lemma (Root): वनम्         | POS: PART
Token: अगच्छत्      | Lemma (Root): अगच्ṣu       | POS: ADJ
Token: ।            | Lemma (Root): una          | POS: NOUN


In [2]:
import stanza

# Initialize the ITB Classical Pipeline
nlp = stanza.Pipeline(
    lang="sa",
    processors={
        "tokenize": "itb",
        "pos": "itb",
        "lemma": "itb",
        "depparse": "itb",
    },
)

# Your clean SLP1 text string
text = "zazakaH nakulaH mArjArI ca"
doc = nlp(text)

for sentence in doc.sentences:
    for word in sentence.words:
        print(
            f"Token: {word.text:<10} | Lemma (Root): {word.lemma:<10} | POS: {word.upos}"
        )

2026-06-28 01:46:16 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-06-28 01:46:16 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-06-28 01:46:16 WARNING: Can not find tokenize: itb from official model list. Ignoring it.
2026-06-28 01:46:16 WARNING: Can not find pos: itb from official model list. Ignoring it.
2026-06-28 01:46:16 WARNING: Can not find lemma: itb from official model list. Ignoring it.
2026-06-28 01:46:16 WARNING: Can not find depparse: itb from official model list. Ignoring it.
2026-06-28 01:46:19 INFO: Loading these models for language: sa (Sanskrit):
| Processor | Package        |
------------------------------
| tokenize  | vedic          |
| pos       | vedic_nocharlm |
| lemma     | vedic_nocharlm |
| depparse  | vedic_nocharlm |

2026-06-28 01:46:19 INFO: Using device: cpu
2026-06-28

Token: zazakaH    | Lemma (Root): zazakah    | POS: NOUN
Token: nakulaH    | Lemma (Root): nakula     | POS: NOUN
Token: mArjArI    | Lemma (Root): marjari    | POS: NOUN
Token: ca         | Lemma (Root): ca         | POS: CCONJ


In [4]:
import stanza

# 1. Download and initialize the correct official Sanskrit package ('vedic')
stanza.download("sa", package="vedic")

nlp = stanza.Pipeline(
    lang="sa",
    processors="tokenize,pos,lemma,depparse",
    package="vedic",
)

# A matching tokenizer pipeline to mirror the exact same cuts on Devanagari
tokenizer_only = stanza.Pipeline(
    lang="sa", 
    processors="tokenize", 
    package="vedic"
)

# Your Data Arrays
transliterated_version = [
    "kAkasya upAyaH",
    "kazcana mahAvRkSaH AsIt |",
    "tatra ekaH kAkaH patnyA saha vasati |"
]

sanskrit_version = [
    "काकस्य उपायः",
    "कश्चन महावृक्षः आसीत् |",
    "तत्र एकः काकः पत्न्या सह वसति |"
]

print("--- Robust Aligned Processing Started ---")

for trans_line, sans_line in zip(transliterated_version, sanskrit_version):
    # Process both strings so they undergo identical compound/punctuation slicing
    doc_slp1 = nlp(trans_line)
    doc_devanagari = tokenizer_only(sans_line)

    # Flatten out both token lists
    slp1_tokens = [word for sent in doc_slp1.sentences for word in sent.words]
    dev_tokens = [token for sent in doc_devanagari.sentences for token in sent.tokens]

    print(f"\nProcessing: {sans_line.strip()}")
    print("-" * 70)

    # Lock step mapping
    for idx, s_word in enumerate(slp1_tokens):
        dev_token_text = (
            dev_tokens[idx].text if idx < len(dev_tokens) else "[Mismatch]"
        )

        print(
            f"Sanskrit: {dev_token_text:<12} | SLP1 Token: {s_word.text:<12} | Lemma: {s_word.lemma:<12} | POS: {s_word.upos}"
        )

2026-06-28 02:01:59 INFO: Downloaded file to /root/.cache/stanza/1.13.0/resources/resources.json
2026-06-28 02:01:59 INFO: Downloading these customized packages for language: sa (Sanskrit)...
| Processor | Package        |
------------------------------
| tokenize  | vedic          |
| pos       | vedic_nocharlm |
| lemma     | vedic_nocharlm |
| depparse  | vedic_nocharlm |
| pretrain  | fasttext157    |

2026-06-28 02:01:59 INFO: File exists: /root/.cache/stanza/1.13.0/resources/sa/tokenize/vedic.pt
2026-06-28 02:01:59 INFO: File exists: /root/.cache/stanza/1.13.0/resources/sa/pos/vedic_nocharlm.pt
2026-06-28 02:01:59 INFO: File exists: /root/.cache/stanza/1.13.0/resources/sa/lemma/vedic_nocharlm.pt
2026-06-28 02:02:00 INFO: File exists: /root/.cache/stanza/1.13.0/resources/sa/depparse/vedic_nocharlm.pt
2026-06-28 02:02:00 INFO: File exists: /root/.cache/stanza/1.13.0/resources/sa/pretrain/fasttext157.pt
2026-06-28 02:02:00 INFO: Finished downloading models and saved to /root/.cache/

--- Robust Aligned Processing Started ---

Processing: काकस्य उपायः
----------------------------------------------------------------------
Sanskrit: काकस्य       | SLP1 Token: kAkasya      | Lemma: kaka         | POS: NOUN
Sanskrit: उपायः        | SLP1 Token: upAyaH       | Lemma: upAyaH       | POS: NOUN

Processing: कश्चन महावृक्षः आसीत् |
----------------------------------------------------------------------
Sanskrit: कश्चन        | SLP1 Token: kazcana      | Lemma: kazcana      | POS: PRON
Sanskrit: महावृक्षः    | SLP1 Token: mahAvRkSaH   | Lemma: mahavrksa    | POS: ADJ
Sanskrit: आसीत्        | SLP1 Token: AsIt         | Lemma: AsIt         | POS: NOUN
Sanskrit: |            | SLP1 Token: |            | Lemma: |            | POS: PART

Processing: तत्र एकः काकः पत्न्या सह वसति |
----------------------------------------------------------------------
Sanskrit: तत्र         | SLP1 Token: tatra        | Lemma: tatra        | POS: ADV
Sanskrit: एकः          | SLP1 Token: ekaH         |

In [1]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import SchemeMap, SCHEMES, transliterate

# 1. Define your Devanagari Sanskrit text
sanskrit_text = "नकुलः गच्छति"

# 2. Transliterate to IAST (Standard academic script with diacritics)
iast_output = transliterate(sanskrit_text, sanscript.DEVANAGARI, sanscript.IAST)

# 3. Transliterate to SLP1 (Sanskrit Library Phonetic basic ASCII encoding)
slp1_output = transliterate(sanskrit_text, sanscript.DEVANAGARI, sanscript.SLP1)

# 4. Transliterate to HK (Harvard-Kyoto capitalization method)
hk_output = transliterate(sanskrit_text, sanscript.DEVANAGARI, sanscript.HK)

print(f"Devanagari : {sanskrit_text}")
print(f"IAST       : {iast_output}")
print(f"SLP1       : {slp1_output}")
print(f"HK         : {hk_output}")


Devanagari : नकुलः गच्छति
IAST       : nakulaḥ gacchati
SLP1       : nakulaH gacCati
HK         : nakulaH gacchati
